In [ ]:
import sys
import os
import numpy as np
# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
# Add the 2nd level parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(parent_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from inflow_model.blade_params import APC_8x6, P600_Blade
from bemt_model import BemtModel
from objective import FittingObjective
from seed_generator import MultiSeedGenerator, SingleSeedGenerator
from fitting_engine import FittingEngine
from fit_plotter import FitPlotter
from manager import FittingManager
from post_processing import make_lookup_table, make_residual_force_columns
from drone import parameters
import inflow_model.propeller_lookup_table as propeller_lookup_table


Prepare data for fitting. 

In [ ]:
import data_factory

# fitting_subfolder = "wind_near_wall_bemt_fitting"
# fitting_subfolder = "wind_near_wall_bemt_fitting_hover"
# fitting_subfolder = "wind_free_space_training"
# fitting_subfolder = "wind_free_space_training_in_house_sim"
# fitting_subfolder = "wind_near_wall_bemt_unit_test"
fitting_subfolder = "wind_free_space_cfd"
factory = data_factory.FittingFactory()
# data_list = data_factory.generate_data_list(fitting_subfolder, '.pkl')
data_list = data_factory.generate_data_list(fitting_subfolder, '.csv')
print(f"Fitting Data list:")
for data in data_list:
    print(data)
datasets = factory.prepare_datasets(data_list)

# fitting_subfolder = "wind_near_wall_bemt_fitting_validation"
# fitting_subfolder = "wind_near_wall_bemt_fitting_far_from_wall_validation"
# fitting_subfolder = "wind_near_wall_bemt_fitting_validation_sinusoidal"
# fitting_subfolder = "wind_near_wall_bemt_fitting_hover_validation"
# fitting_subfolder = "wind_near_wall_bemt_fitting_hover_far_from_wall_validation"
# fitting_subfolder = "wind_free_space_validation"
# fitting_subfolder = "wind_free_space_validation_in_house_sim"
fitting_subfolder = "wind_free_space_validation_cfd"
factory = data_factory.FittingFactory()
# data_list = data_factory.generate_data_list(fitting_subfolder, '.pkl')
data_list = data_factory.generate_data_list(fitting_subfolder, '.csv')
print(f"Validtion Data list:")
for data in data_list:
    print(data)
datasets_validation = factory.prepare_datasets(data_list)

Start fitting.

In [ ]:
is_multiseed = False
is_fine_tune = True
# init_guess = [37.406, 31.987, 3.512, 0.481, 0.000]  # cl_1=37.406  cl_2=31.987  cd=3.512  alpha_0=27.566deg  k_body_drag=0.000
init_guess = [3.723, 18.819, 4.977, 0.693838191, 0.3763]  # cl_1=3.723  cl_2=18.819  cd=4.977  alpha_0=39.754deg  k_body_drag=0.3763
init_guess = [8.920, 36.263, 4.166, 0.481, 0.016]  # cl_1=8.920  cl_2=36.263  cd=4.166  alpha_0=27.595deg  k_body_drag=0.016
# cl_1=8.088  cl_2=16.060  cd=5.000  alpha_0=40.000deg  k_body_drag=0.000
manager = FittingManager.for_full_vehicle(P600_Blade(), parameters.P600(), datasets, init_guess=init_guess)
manager.run(is_multiseed=is_multiseed, is_fine_tune=is_fine_tune)


To test the fitted params, one can directly apply the fitted solution to BEMT model to generate result to compare with the label. Use sparse samples as this is computationally expensive. 

In [ ]:
model = BemtModel(APC_8x6(), parameters.PennStateARILab550())
x = np.array([5.3, 1.7, 1.8, np.radians(20.6), 0.0])  # cl_1, cl_2, cd, alpha_0, k_body_drag
model.adjust_resolution(True)
# lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("apc_8x6_with_trail_refine")
# lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("apc_8x6_zero")
# lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("apc_8x6_fitted_in_noise_and_vibration")
lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("p600_full_range")
objective = FittingObjective(model)
objective.get_loss(x, datasets[:5], lookup_table=lookup_table, is_using_lookup_table=True)


To do a thorough test, generate a lookup table using inflow_model.propeller_lookup_table_users_guide.ipynb and come back here to check the loss over all sammple data.

In [ ]:
model = BemtModel(P600_Blade(), parameters.P600())
model.adjust_resolution(True)
make_lookup_table([3.723, 18.819, 4.977, 0.693838191], P600_Blade(), "p600_full_range", is_hover_only=False)

In [ ]:
import inflow_model.propeller_lookup_table as propeller_lookup_table
lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("p600_full_range")
manager = FittingManager.for_full_vehicle(P600_Blade(), parameters.P600(), datasets)
fig0, fig1 = manager.plot(dataset_idx=1, lookup_table=lookup_table, is_using_lookup_table=True, sample_step=5)

Append the residual force calculated based on the lookup table to the end of the dataset. In the next step, this augmented dataset will be used for network training.

In [ ]:
model = BemtModel(APC_8x6(), parameters.PennStateARILab550())
# lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("apc_8x6_fitted_in_noise_and_vibration")
# lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("apc_8x6_fitted_in_noise_and_vibration_hover")
lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("p600_full_range")
for dataset in datasets:
    print(f"Filling residual force column of {os.path.relpath(dataset.path_to_data_file)}")
    make_residual_force_columns(model, dataset, lookup_table)


Similarly, append the residual force to validation dataset as well for validation after training.

In [ ]:
model = BemtModel(APC_8x6(), parameters.PennStateARILab550())
# lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("apc_8x6_fitted_in_noise_and_vibration")
# lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("apc_8x6_fitted_in_noise_and_vibration_hover")
lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("p600_full_range")
for dataset in datasets_validation:
    print(f"Filling residual force column of {os.path.relpath(dataset.path_to_data_file)}")
    make_residual_force_columns(model, dataset, lookup_table)